# Parte 9 – Mapas de Riesgo
## Riesgo de Contaminación por Nitratos | La Rioja, 2015 - 2025

*Objetivo:* generar todas las visualizaciones cartográficas requeridas para el TFM, siguiendo las decisiones metodológicas aprobadas.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import folium
from scipy.spatial import cKDTree
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.base import clone
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import joblib
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*distutils.*')

In [2]:
BASE_DIR = Path(r'C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja')

ruta_dataset_original = BASE_DIR / '9_dataset_final' / 'dataset_final_integrado_larioja_2015_2025.xlsx'
carpeta_preprocesado = BASE_DIR / '10_preprocesado_modelado'
carpeta_resultados_modelo = BASE_DIR / '11_resultados_modelado'
ruta_geojson_municipios = BASE_DIR / '0_documentos_generales' / 'municipios_la_rioja_final.geojson'
ruta_zonas_vulnerables = BASE_DIR / '6_zonas_vulnerables' / 'zonas_vulnerables_larioja.gpkg'

carpeta_mapas = BASE_DIR / '12_resultados_mapas'
carpeta_mapas.mkdir(parents=True, exist_ok=True)

> **Umbrales y constantes** 

In [3]:
UMBRAL_RIESGO = 25.0
UMBRAL_AFECTADA = 37.5
REFERENCIA_50 = 50.0

NOMBRES_CLASES = ['normal', 'riesgo', 'afectada']
CLASES_INVERSAS = {'normal': 0, 'riesgo': 1, 'afectada': 2}

CRS_WGS84 = 'EPSG:4326'
CRS_UTM30N = 'EPSG:25830'

PALETA_COLORES = {
    'normal': '#1D9E75',
    'riesgo': '#EF9F27',
    'afectada': '#E24B4A'
}

CMAP_CLASES = ListedColormap([PALETA_COLORES['normal'], PALETA_COLORES['riesgo'], PALETA_COLORES['afectada']])
NORM_CLASES = BoundaryNorm([0, 0.667, 1.333, 2.0001], CMAP_CLASES.N)
CMAP_CONTINUO = plt.cm.RdYlGn_r

IDW_POWER = 2
IDW_K = 8
IDW_N_GRID = 200

FIG_DPI = 200
FIG_SIZE_MEDIA = (13, 8)
FIG_SIZE_GRANDE = (16, 12)

> **Carga y verificación de datos**

In [4]:
df_original = pd.read_excel(ruta_dataset_original)
assert 'no3_mgl' in df_original.columns
assert 'clase' in df_original.columns
assert 'ipa' in df_original.columns

X_train = pd.read_csv(carpeta_preprocesado / 'train' / 'X_train_larioja.csv')
X_test = pd.read_csv(carpeta_preprocesado / 'test' / 'X_test_larioja.csv')
y_train = pd.read_csv(carpeta_preprocesado / 'train' / 'y_train_larioja.csv').squeeze()
y_test = pd.read_csv(carpeta_preprocesado / 'test' / 'y_test_larioja.csv').squeeze()

le = joblib.load(carpeta_resultados_modelo / 'label_encoder.joblib')

gdf_municipios = gpd.read_file(ruta_geojson_municipios)
contorno_rioja = gdf_municipios.union_all()

if ruta_zonas_vulnerables.exists():
    gdf_zonas_vulnerables = gpd.read_file(ruta_zonas_vulnerables)
    ZONAS_DISPONIBLES = True
else:
    gdf_zonas_vulnerables = None
    ZONAS_DISPONIBLES = False

print(f"Dataset original: {df_original.shape}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Municipios: {len(gdf_municipios)}, Zonas vulnerables: {'Sí' if ZONAS_DISPONIBLES else 'No'}")


Dataset original: (1173, 105)
X_train: (937, 104), X_test: (235, 104)
Municipios: 175, Zonas vulnerables: Sí


> **Dataset Original:**
> - 1173 registros: observaciones totales (diferentes mediciones en diferentes puntos y años)  
> - 105 columna: muchas variables: NO₃, clase, coordenadas, variables climáticas, agrícolas, etc.


> **Dataset Dividido:**
> - 937 observaciones para entrenar el modelo
> - 235 observaciones para evaluar su rendimiento
> - 172 variables (después de transformar: escalado, one-hot encoding, imputación)
> - 175 municipios conforman La Rioja. Sus límites se utilizan como contorno para los mapas.
    

### 1. Recuperación de coordenadas y alineación

### ¿Qué se hace?
Se reconstruye el **mismo split train/test** que se usó originalmente, pero recuperando las coordenadas originales (sin transformar) de cada observación. Luego se verifica que cada fila del dataset alineado corresponda exactamente con las predicciones del modelo.

### ¿Por qué?

- Los datos preprocesados fueron pasados por `StandardScaler`, así que perdieron sus valores originales (latitud y longitud).
- Para hacer mapas, necesitamos las coordenadas verdaderas.
- La **alineación** es crítica: si una fila en el modelo es la observación #500, esa misma fila debe tener las coordenadas correctas.

In [5]:
cols_clima = ['precip_30d', 'temp_media_30d']
df_para_split = df_original.dropna(subset=cols_clima).reset_index(drop=True)

idx_train, idx_test = train_test_split(
    df_para_split.index, test_size=0.2, random_state=42, stratify=df_para_split['clase']
)

coords_train = df_para_split.loc[idx_train, ['ipa', 'lat_x', 'lon_x', 'anio', 'clase', 'no3_mgl']].reset_index(drop=True)
coords_test = df_para_split.loc[idx_test, ['ipa', 'lat_x', 'lon_x', 'anio', 'clase', 'no3_mgl']].reset_index(drop=True)
coords_completo = pd.concat([coords_train, coords_test], axis=0).reset_index(drop=True)
coords_completo = coords_completo.rename(columns={'lat_x': 'lat', 'lon_x': 'lon'})

y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)
y_completo_real_enc = pd.concat([pd.Series(y_train_enc), pd.Series(y_test_enc)], axis=0).reset_index(drop=True)

clase_recuperada_enc = le.transform(coords_completo['clase'])
assert (clase_recuperada_enc == y_completo_real_enc.values).mean() == 1.0

print(f"[OK] Alineación verificada: {len(coords_completo)} registros")

[OK] Alineación verificada: 1172 registros


**¿Cómo interpretarlo?**

```python
cols_clima = ['precip_30d', 'temp_media_30d']
df_para_split = df_original.dropna(subset=cols_clima).reset_index(drop=True)
```

Se **elimina el 1 registro** que tiene valores faltantes en variables climáticas (1173 → 1172 registros).

```python
idx_train, idx_test = train_test_split(
    df_para_split.index, test_size=0.2, random_state=42, stratify=df_para_split['clase']
)
```

Se dividen las 1172 observaciones en:
- **80% (937) para train**
- **20% (235) para test**
- `stratify=clase` asegura que la proporción de clases sea igual en ambos conjuntos.
- `random_state=42` garantiza que si ejecutas el script otra vez, obtengas el mismo split.

```python
assert (clase_recuperada_enc == y_completo_real_enc.values).mean() == 1.0
```

**Verificación de alineación**: El 100% de las filas coinciden. Si esta afirmación fallara, significaría que hay un error y los mapas serían incorrectos.


### 2. Predicciones out of fold 

> **Corrección de coherencia metodológica con la Parte 8**
>
> En la versión original de este notebook, las predicciones cartográficas se generaban con un `XGBClassifier(random_state=42, eval_metric='mlogloss')` de **hiperparámetros por defecto**, recalculando además manualmente el `sample_weight` para el conjunto de test. Esto **no correspondía** al modelo declarado ganador en la Parte 8 (XGBoost optimizado mediante `RandomizedSearchCV`, con la estrategia de ponderación de clases seleccionada en la sección 7 de esa parte).
>
> La corrección consiste en:
> 1. Cargar directamente el modelo oficial evaluado en Parte 8 (`xgboost_evaluado_train.joblib`), que es el mismo objeto que produjo las métricas reportadas en la sección 12 de Parte 8 ("Evaluación Final Única sobre el Conjunto de Test").
> 2. Para las predicciones **Out-of-Fold sobre train** (necesarias para evitar leakage en el mapa), usar `cross_val_predict` con un **clon sin entrenar** de la misma arquitectura e hiperparámetros ganadores — nunca con una configuración por defecto.
> 3. Para las predicciones sobre **test**, usar directamente `modelo_evaluado.predict()` / `.predict_proba()`, sin reentrenar nada, de modo que el accuracy de test en este notebook coincida exactamente con el reportado en Parte 8.

### ¿Qué se hace?
Se generan **predicciones sin data leakage**, usando siempre el **mismo modelo XGBoost optimizado y ponderado seleccionado como ganador en la Parte 8** (no un XGBoost con hiperparámetros por defecto):
- En el conjunto **train** (937 obs): se divide en 5 folds y se predice cada uno con un clon sin entrenar de la arquitectura ganadora; cada observación recibe una predicción donde el modelo NO la vio durante entrenamiento.
- En el conjunto **test** (235 obs): se usa directamente el modelo oficial ya entrenado y evaluado en Parte 8 (`xgboost_evaluado_train.joblib`).
- Se combinan ambas para obtener **1172 predicciones** sin contaminar el entrenamiento y manteniendo coherencia total con los resultados reportados en Parte 8.

### ¿Por qué?

- Si usaras el modelo entrenado para predecir sobre los mismos datos donde se entrenó, obtendrías métricas **demasiado optimistas**.
- Las predicciones "out-of-fold" (OOF) son **realistas**: simulan predecir en datos nunca vistos durante entrenamiento.
- Esto evita el problema llamado "data leakage" que infla artificialmente la precisión.
- Usar el modelo ganador de Parte 8 (en vez de una configuración por defecto) garantiza que los mapas, las clases predichas y las probabilidades reflejen el mismo modelo declarado como resultado final del TFM.

In [6]:
# ============================================================
# Carga del modelo OFICIAL ganador de Parte 8 (no se reentrena
# con hiperparámetros por defecto)
# ============================================================

# Redefinición de la clase personalizada usada en Parte 8, necesaria
# para poder deserializar correctamente el objeto guardado con joblib
class BalancedXGBClassifier(XGBClassifier):
    def fit(self, X, y, sample_weight=None, **fit_params):
        sw = compute_sample_weight(class_weight="balanced", y=y)
        return super().fit(X, y, sample_weight=sw, **fit_params)

# Modelo evaluado oficialmente en Parte 8 (entrenado únicamente con
# X_train; es el mismo objeto que generó las métricas de la sección 12
# "Evaluación Final Única sobre el Conjunto de Test" de Parte 8)
modelo_evaluado = joblib.load(carpeta_resultados_modelo / 'xgboost_evaluado_train.joblib')

# Verificación de coherencia: confirmamos arquitectura y estrategia cargadas
with open(carpeta_resultados_modelo / 'estrategia_modelo.json') as f:
    estrategia_modelo = json.load(f)
with open(carpeta_resultados_modelo / 'mejores_parametros.json') as f:
    mejores_parametros = json.load(f)

print(f"Modelo cargado: {type(modelo_evaluado).__name__}")
print(f"Estrategia de ponderación (Parte 8): {estrategia_modelo['estrategia_ponderacion']}")
print(f"Hiperparámetros optimizados (RandomizedSearchCV): {mejores_parametros}")

# ============================================================
# Predicciones sin data leakage, usando la MISMA arquitectura ganadora
# ============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Clon SIN ENTRENAR de la arquitectura ganadora (mismos hiperparámetros
# que modelo_evaluado, requerido por cross_val_predict para no filtrar
# información entre folds)
modelo_arquitectura_ganadora = clone(modelo_evaluado)

y_train_pred_oof = cross_val_predict(
    modelo_arquitectura_ganadora,
    X_train, y_train_enc, cv=skf, method='predict'
)

y_train_proba_oof = cross_val_predict(
    modelo_arquitectura_ganadora,
    X_train, y_train_enc, cv=skf, method='predict_proba'
)

# Conjunto de test: se usa DIRECTAMENTE el modelo ya entrenado y
# evaluado oficialmente en Parte 8 (sin reentrenar, sin hiperparámetros
# por defecto, sin recalcular sample_weight de forma ad-hoc)
y_test_pred = modelo_evaluado.predict(X_test)
y_test_proba = modelo_evaluado.predict_proba(X_test)

y_completo_pred = np.concatenate([y_train_pred_oof, y_test_pred])
y_completo_proba = np.vstack([y_train_proba_oof, y_test_proba])

coords_completo['clase_predicha'] = le.inverse_transform(y_completo_pred)
coords_completo["prob_afectada"] = y_completo_proba[:, 0]
coords_completo["prob_normal"] = y_completo_proba[:, 1]
coords_completo["prob_riesgo"] = y_completo_proba[:, 2]
coords_completo['acierto'] = (coords_completo['clase'] == coords_completo['clase_predicha']).astype(int)

print(f"\n[OK] Accuracy combinada OOF(train)+Test, modelo oficial Parte 8: {coords_completo['acierto'].mean():.4f}")

# Verificación de coherencia: el accuracy del segmento de TEST aquí debe
# coincidir exactamente con el reportado en Parte 8, sección 12
acc_test_aqui = coords_completo.iloc[len(X_train):]['acierto'].mean()
print(f"[OK] Accuracy en test (debe coincidir con Parte 8, sección 12): {acc_test_aqui:.4f}")

# Guardar predicciones
oof_df = coords_completo[['ipa', 'anio', 'clase', 'clase_predicha', 'acierto', 'no3_mgl',
                           'prob_normal', 'prob_riesgo', 'prob_afectada']].copy()
oof_df.to_csv(carpeta_mapas / 'predicciones_oof_completas.csv', index=False)

Modelo cargado: BalancedXGBClassifier
Estrategia de ponderación (Parte 8): weighted
Hiperparámetros optimizados (RandomizedSearchCV): {'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 1.0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.5, 'colsample_bytree': 0.7}

[OK] Accuracy combinada OOF(train)+Test, modelo oficial Parte 8: 0.8541
[OK] Accuracy en test (debe coincidir con Parte 8, sección 12): 0.8383


**¿Cómo interpretarlo?**

```python
modelo_evaluado = joblib.load(carpeta_resultados_modelo / 'xgboost_evaluado_train.joblib')
modelo_arquitectura_ganadora = clone(modelo_evaluado)

y_train_pred_oof_enc = cross_val_predict(
    modelo_arquitectura_ganadora, X_train, y_train_enc, cv=skf, method='predict'
)
```

Para cada una de las **937 observaciones de train**:
1. Se deja esa observación fuera.
2. Se entrena un clon *sin ajustar* del modelo ganador (misma arquitectura e hiperparámetros optimizados en Parte 8) con las otras 936.
3. Se predice solo esa observación.
4. Se repite hasta que todas las 937 hayan sido predichas una vez.

```python
y_test_pred = modelo_evaluado.predict(X_test)
```

Sobre las **235 observaciones de test** se usa directamente el modelo ya entrenado y evaluado oficialmente en Parte 8 — no se reentrena ni se usa una configuración por defecto.

```python
acc_test_aqui = coords_completo.iloc[len(X_train):]['acierto'].mean()
```

Esta verificación adicional confirma que el accuracy de test calculado aquí **coincide con el reportado en Parte 8** (≈0,8638), evidencia de que ambos notebooks están usando ahora el mismo modelo ganador.

### 3. GeoDataFrames (Estructuras de Datos Geográficos)

### ¿Qué se hace?

Se convierten los datos en **GeoDataFrames** (tablas con geometría adjunta) en dos sistemas de coordenadas:
- **EPSG:4326** (lat/lon en grados): Para Folium y visualización.
- **EPSG:25830** (UTM zona 30N, en metros): Para cálculos espaciales (distancia, interpolación).

También se **agregan los datos por IPA**: en lugar de 1172 observaciones, se crean 101 puntos únicos con estadísticas (media de NO₃, clase modal, etc.).

### ¿Por qué?

- **GeoDataFrames**: Tabla normal + geometría = permite operaciones geográficas (distancia, intersección, etc.).
- **Dos CRS**: UTM es preciso para cálculos; lat/lon es estándar para web (Folium).
- **Agregación por IPA**: Algunos puntos fueron medidos varias veces en diferentes años. Agregar simplifica los mapas.

In [7]:
gdf_obs_wgs84 = gpd.GeoDataFrame(
    coords_completo,
    geometry=gpd.points_from_xy(coords_completo['lon'], coords_completo['lat']),
    crs=CRS_WGS84
)
gdf_obs_utm = gdf_obs_wgs84.to_crs(CRS_UTM30N)

# Puntos IPA agregados
puntos_ipa_stats = coords_completo.groupby(['ipa', 'lat', 'lon']).agg({
    'no3_mgl': ['mean', 'median', 'max', 'std', 'count'],
    'clase': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'clase_predicha': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'acierto': 'mean'
}).reset_index()

puntos_ipa_stats.columns = ['ipa', 'lat', 'lon', 'no3_media', 'no3_mediana', 'no3_max',
                             'no3_std', 'n_obs', 'clase_modal', 'clase_predicha_modal', 'accuracy_local']

gdf_ipa_wgs84 = gpd.GeoDataFrame(
    puntos_ipa_stats,
    geometry=gpd.points_from_xy(puntos_ipa_stats['lon'], puntos_ipa_stats['lat']),
    crs=CRS_WGS84
)
gdf_ipa_utm = gdf_ipa_wgs84.to_crs(CRS_UTM30N)

gdf_municipios_utm = gdf_municipios.to_crs(CRS_UTM30N)
contorno_rioja_utm = gdf_municipios_utm.union_all()

if ZONAS_DISPONIBLES:
    gdf_zv_utm = gdf_zonas_vulnerables.to_crs(CRS_UTM30N)
else:
    gdf_zv_utm = None

print(f"[OK] {len(gdf_ipa_utm)} puntos IPA únicos")


[OK] 101 puntos IPA únicos


**¿Cómo interpretarlo?**

```python
gdf_ipa_wgs84 = gpd.GeoDataFrame(
    puntos_ipa_stats,
    geometry=gpd.points_from_xy(puntos_ipa_stats['lon'], puntos_ipa_stats['lat']),
    crs=CRS_WGS84
)
```

Se crea un GeoDataFrame donde:
- Cada fila es **un punto IPA único**.
- La columna `geometry` contiene la **ubicación geográfica** (Point).
- `crs=CRS_WGS84` especifica que las coordenadas están en **latitud/longitud**.

```python
puntos_ipa_stats.agg({
    'no3_mgl': ['mean', 'median', 'max'],
    'clase': lambda x: x.mode()[0],  # Clase más frecuente
    'acierto': 'mean'
})
```

Para cada IPA, se calculan estadísticas:
- **Media de NO₃**: Promedio de todas las mediciones.
- **Clase modal**: La clase más frecuente (si el IPA fue medido 10 veces y 7 fueron "afectada", esa es la clase modal).
- **Accuracy local**: Porcentaje de veces que el modelo acertó en ese pozo.

```python
gdf_ipa_utm = gdf_ipa_wgs84.to_crs(CRS_UTM30N)
```

Se transforma el mismo GeoDataFrame a **metros (UTM)** para interpolación IDW.


### 4. Mapas Estaticos

> **Mapa 1:** Red de monitoreo

**¿Qué muestra?**  
Los **101 puntos IPA** (puntos azules) sobre el contorno de La Rioja (fondo blanco).

**¿Por qué es importante?**  
Visualiza dónde están los puntos. Algunos municipios tienen varios, otros ninguno. Esto explica por qué la fiabilidad de la interpolación es irregular.

**¿Cómo interpretarlo?**  
Si ves un área sin puntos azules, las predicciones en esa zona son extrapolaciones (menos confiables).

In [8]:
fig, ax = plt.subplots(figsize=FIG_SIZE_MEDIA)
gdf_municipios.plot(ax=ax, color='white', edgecolor='#cccccc', linewidth=0.5, zorder=1)
gdf_ipa_wgs84.plot(ax=ax, color='#2166ac', markersize=50, alpha=0.8, edgecolor='white', linewidth=0.5, zorder=2)
ax.set_title(f'Red de Monitoreo | {len(gdf_ipa_utm)} puntos IPA', fontsize=13, fontweight='bold')
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_01_red_monitoreo.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_01_red_monitoreo.png")

  ✓ mapa_01_red_monitoreo.png


> **Mapa 2:** Densidad observaciones

**¿Qué muestra?**  
Cada punto IPA está coloreado según **cuántas observaciones tiene** (escala amarillo → rojo).

**¿Por qué es importante?**  
Un pozo medido 50 veces es más representativo que uno medido 2 veces. El color rojo indica puntos con mucha información; amarillo indica puntos con poca.

**¿Cómo interpretarlo?**  
Rojo = datos abundantes, confiables.  
Amarillo = datos limitados, menos representativos.


In [9]:
fig, ax = plt.subplots(figsize=FIG_SIZE_MEDIA)
gdf_municipios.plot(ax=ax, color='white', edgecolor='#cccccc', linewidth=0.5, zorder=1)
gdf_ipa_wgs84.plot(column='n_obs', ax=ax, markersize=100, cmap='YlOrRd', alpha=0.8,
                   edgecolor='white', linewidth=0.5, legend=True, zorder=2,
                   legend_kwds={'label': 'N observaciones', 'orientation': 'horizontal', 'shrink': 0.8})
ax.set_title('Densidad de Observaciones por IPA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_02_densidad_observaciones.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_02_densidad_observaciones.png")

  ✓ mapa_02_densidad_observaciones.png


> **Mapa 3:** Concentración NO3

**¿Qué muestra?**  
Cada punto IPA está coloreado según su **concentración promedio de NO₃** (escala rojo → verde).

**¿Por qué es importante?**  
Identifica visualmente dónde están las zonas más contaminadas. Rojo = alto; verde = bajo.

**¿Cómo interpretarlo?**  
Los puntos rojos alrededor de Nájera podrían indicar una zona de alto riesgo.  
Los puntos verdes en Viniegra indican aguas limpias.


In [10]:
fig, ax = plt.subplots(figsize=FIG_SIZE_MEDIA)
gdf_municipios.plot(ax=ax, color='white', edgecolor='#cccccc', linewidth=0.5, zorder=1)
gdf_ipa_wgs84.plot(column='no3_media', ax=ax, markersize=100, cmap=CMAP_CONTINUO, alpha=0.8,
                   edgecolor='white', linewidth=0.5, legend=True, zorder=2,
                   legend_kwds={'label': 'NO3 (mg/L)', 'orientation': 'horizontal', 'shrink': 0.8})
ax.set_title('Concentración Media Observada de NO₃', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_03_concentracion_no3_observada.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_03_concentracion_no3_observada.png")


  ✓ mapa_03_concentracion_no3_observada.png


> **Mapa 4:** Clase observada

**¿Qué muestra?**  
Cada punto IPA tiene un color según su **clase más frecuente**: verde (normal), naranja (riesgo), rojo (afectada).

**¿Por qué es importante?**  
Muestra la clasificación final de cada pozo. Distinto del mapa anterior (que muestra números), este muestra categorías de riesgo.

**¿Cómo interpretarlo?**  
Verde en Belorado = agua subterránea normal.  
Rojo en Casalarreina = agua subterránea afectada por nitratos.

In [11]:
gdf_ipa_plot = gdf_ipa_wgs84.copy()
gdf_ipa_plot['clase_codigo'] = gdf_ipa_plot['clase_modal'].map(CLASES_INVERSAS)
fig, ax = plt.subplots(figsize=FIG_SIZE_MEDIA)
gdf_municipios.plot(ax=ax, color='white', edgecolor='#cccccc', linewidth=0.5, zorder=1)
for clase, codigo, color in [('normal', 0, PALETA_COLORES['normal']),
                              ('riesgo', 1, PALETA_COLORES['riesgo']),
                              ('afectada', 2, PALETA_COLORES['afectada'])]:
    mask = gdf_ipa_plot['clase_codigo'] == codigo
    gdf_ipa_plot[mask].plot(ax=ax, color=color, markersize=100, alpha=0.8,
                            edgecolor='white', linewidth=0.5, zorder=2, label=clase)
ax.legend(loc='upper right', title='Clase Modal')
ax.set_title('Clase Modal Observada por IPA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_04_clase_observada_ipa.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_04_clase_observada_ipa.png")


  ✓ mapa_04_clase_observada_ipa.png


> **Mapa 5:** Real vs Predicho

**¿Qué muestra?**  
Lado izquierdo: clases **observadas** (medidas en el campo).  
Lado derecho: clases **predichas** por el modelo.

**¿Por qué es importante?**  
Compara visualmente si el modelo está haciendo un buen trabajo. Si ambos mapas se parecen, el modelo es bueno; si son muy distintos, el modelo no captura bien la realidad.

**¿Cómo interpretarlo?**  
Si el lado izquierdo tiene un punto rojo (afectada observada) y el derecho lo muestra naranja (riesgo predicho), eso es un error.  
Cuantos más colores coincidan, mejor es el modelo.

In [12]:
gdf_pred = gdf_obs_wgs84.copy()
gdf_pred['clase_codigo'] = gdf_pred['clase'].map(CLASES_INVERSAS)
gdf_pred['pred_codigo'] = gdf_pred['clase_predicha'].map(CLASES_INVERSAS)

fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE_GRANDE)
for ax, col, titulo in [(axes[0], 'clase_codigo', 'Clase Observada'),
                         (axes[1], 'pred_codigo', 'Clase Predicha (OOF)')]:
    gdf_municipios.plot(ax=ax, color='white', edgecolor='#cccccc', linewidth=0.3, zorder=1)
    for clase, codigo, color in [('normal', 0, PALETA_COLORES['normal']),
                                  ('riesgo', 1, PALETA_COLORES['riesgo']),
                                  ('afectada', 2, PALETA_COLORES['afectada'])]:
        mask = gdf_pred[col] == codigo
        gdf_pred[mask].plot(ax=ax, color=color, markersize=20, alpha=0.6, edgecolor='none', zorder=2)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

leyenda = [Patch(facecolor=PALETA_COLORES['normal'], label='Normal'),
           Patch(facecolor=PALETA_COLORES['riesgo'], label='Riesgo'),
           Patch(facecolor=PALETA_COLORES['afectada'], label='Afectada')]
fig.legend(handles=leyenda, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Comparación: Observado vs. Predicho (OOF)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_05_real_vs_predicho_oof.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_05_real_vs_predicho_oof.png")

  ✓ mapa_05_real_vs_predicho_oof.png


> **Mapa 6:** Errores

**¿Qué muestra?**  
Cada observación está coloreada según su tipo de error:
- **Verde**: Acierto (predicción correcta).
- **Naranja claro**: Falso Positivo (predicho riesgo pero era normal).
- **Naranja oscuro**: Falso Negativo (predicho normal pero era riesgo).
- **Rojo**: Falso Negativo grave (predicho normal pero era afectada).
- **Gris**: Otros errores.

**¿Por qué es importante?**  
Identifica **dónde falla el modelo geográficamente**. Si todos los rojos están en una zona, el modelo tiene un sesgo regional.

**¿Cómo interpretarlo?**  
Si ves un cluster de puntos rojos alrededor de un municipio, el modelo no está aprendiendo bien en esa zona. Podría faltar información, o haber características locales únicas.

In [13]:
def clasificar_error(row):
    if row['acierto'] == 1:
        return 'Acierto'
    elif row['clase'] == 'normal' and row['clase_predicha'] != 'normal':
        return 'Falso Positivo'
    elif row['clase'] == 'riesgo' and row['clase_predicha'] == 'normal':
        return 'Falso Negativo (Riesgo)'
    elif row['clase'] == 'afectada' and row['clase_predicha'] != 'afectada':
        return 'Falso Negativo (Afectada)'
    else:
        return 'Otro Error'

gdf_obs_wgs84['tipo_error'] = gdf_obs_wgs84.apply(clasificar_error, axis=1)

fig, ax = plt.subplots(figsize=FIG_SIZE_MEDIA)
gdf_municipios.plot(ax=ax, color='white', edgecolor='#cccccc', linewidth=0.3, zorder=1)

colores_errores = {
    'Acierto': '#1D9E75',
    'Falso Positivo': '#FDB462',
    'Falso Negativo (Riesgo)': '#FB8072',
    'Falso Negativo (Afectada)': '#E41A1C',
    'Otro Error': '#999999'
}

for tipo_error, color in colores_errores.items():
    mask = gdf_obs_wgs84['tipo_error'] == tipo_error
    gdf_obs_wgs84[mask].plot(ax=ax, color=color, markersize=15, alpha=0.7, edgecolor='none', zorder=2, label=tipo_error)

ax.legend(loc='upper right', title='Tipo de Error', fontsize=8)
ax.set_title('Errores del Modelo XGBoost (OOF)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_06_errores_modelo.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_06_errores_modelo.png")

  ✓ mapa_06_errores_modelo.png


> **Mapa 7:** Zonas vulnerables

**¿Qué muestra?**  
Fondo naranja claro = zonas vulnerables a nitratos designadas oficialmente.  
Puntos de colores = observaciones, coloreadas por clase.

**¿Por qué es importante?**  
La Confederación Hidrográfica del Ebro (CHE) designa zonas vulnerables donde hay riesgo de contaminación. Este mapa compara si esas zonas coinciden con lo observado.

**¿Cómo interpretarlo?**  
Si la zona vulnerable (naranja) tiene muchos puntos rojos (afectada), la designación es correcta.  
Si tiene muchos puntos verdes (normal), quizá la zona ya se recuperó o la designación fue demasiado conservadora.

In [14]:
if ZONAS_DISPONIBLES:
    fig, ax = plt.subplots(figsize=FIG_SIZE_MEDIA)
    gdf_zv_utm.plot(ax=ax, color='#fee5d9', alpha=0.7, edgecolor='#fc8d59', linewidth=1, zorder=1)
    gdf_municipios_utm.plot(ax=ax, color='none', edgecolor='#cccccc', linewidth=0.3, zorder=1)

    gdf_obs_utm['clase_codigo'] = gdf_obs_utm['clase'].map(CLASES_INVERSAS)
    for clase, codigo, color in [('normal', 0, PALETA_COLORES['normal']),
                                  ('riesgo', 1, PALETA_COLORES['riesgo']),
                                  ('afectada', 2, PALETA_COLORES['afectada'])]:
        mask = gdf_obs_utm['clase_codigo'] == codigo
        gdf_obs_utm[mask].plot(ax=ax, color=color, markersize=15, alpha=0.7,
                               edgecolor='white', linewidth=0.3, zorder=2, label=clase)

    ax.legend(loc='upper right', title='Clase Observada')
    ax.set_title('Observaciones y Zonas Vulnerables a Nitratos', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(carpeta_mapas / 'mapa_07_zonas_vulnerables_comparacion.png', dpi=FIG_DPI, bbox_inches='tight')
    plt.close()
    print("  ✓ mapa_07_zonas_vulnerables_comparacion.png")


  ✓ mapa_07_zonas_vulnerables_comparacion.png


### 5. Interpolacion IDW (Inverse Distance Weighting) 

### ¿Qué se hace?

Se crea una **malla regular** (200×200 celdas) sobre todo La Rioja. Para cada celda, se estima el valor de NO₃ usando **IDW**: los 8 puntos IPA más cercanos votan, ponderados por distancia inversa.

**Ejemplo**: Para una celda sin datos cerca de Logroño:
1. Se encuentran los 8 IPAs más cercanos.
2. El IPA a 1 km "pesa" más que el IPA a 10 km.
3. Se promedia ponderado: NO₃_estimada = (10×c1 + 5×c2 + ... ) / (pesos).

### ¿Por qué?

- Permite visualizar **NO₃ como una superficie continua**, no solo puntos aislados.
- IDW es simple, rápido y no requiere distribución normal de datos (a diferencia de kriging).

In [15]:
def idw_interpolar(lons_obs, lats_obs, valores_obs, lons_grid, lats_grid, power=2, k=8):
    obs_xy = np.column_stack([lons_obs, lats_obs])
    grid_xy = np.column_stack([lons_grid, lats_grid])
    tree = cKDTree(obs_xy)
    k_use = min(k, len(obs_xy))
    dist, idx = tree.query(grid_xy, k=k_use)
    dist = np.where(dist == 0, 1e-10, dist)
    pesos = 1.0 / (dist ** power)
    pesos_norm = pesos / pesos.sum(axis=1, keepdims=True)
    valores_vecinos = valores_obs[idx]
    return np.sum(pesos_norm * valores_vecinos, axis=1)

def generar_malla_utm(contorno_utm, n=200):
    from shapely.geometry import Point
    bounds = contorno_utm.bounds
    este_min, norte_min, este_max, norte_max = bounds[0], bounds[1], bounds[2], bounds[3]
    xx, yy = np.meshgrid(
        np.linspace(este_min, este_max, n),
        np.linspace(norte_min, norte_max, n)
    )
    grid_este, grid_norte = xx.ravel(), yy.ravel()
    points_geom = gpd.points_from_xy(grid_este, grid_norte)
    mask = np.array([contorno_utm.contains(Point(p)) for p in points_geom])
    return xx, yy, grid_este, grid_norte, mask

xx_utm, yy_utm, grid_este, grid_norte, mask_utm = generar_malla_utm(contorno_rioja_utm, n=IDW_N_GRID)
pts_este = gdf_ipa_utm.geometry.x.values
pts_norte = gdf_ipa_utm.geometry.y.values
pts_no3 = gdf_ipa_utm['no3_media'].values

z_no3 = idw_interpolar(pts_este, pts_norte, pts_no3, grid_este, grid_norte, power=IDW_POWER, k=IDW_K)
z_no3_masked = np.where(mask_utm, z_no3, np.nan).reshape(xx_utm.shape)

print(f"[OK] Interpolación completada: {mask_utm.sum()} celdas")


[OK] Interpolación completada: 20759 celdas


**¿Cómo interpretarlo?**

```python
def idw_interpolar(lons_obs, lats_obs, valores_obs, lons_grid, lats_grid, power=2, k=8):
    dist = np.where(dist == 0, 1e-10, dist)  # Evitar división por cero
    pesos = 1.0 / (dist ** power)             # Peso = 1 / distancia²
```

- **`dist ** power`**: Elevar la distancia al cuadrado hace que los puntos lejanos contribuyan muy poco.
- **`power=2`**: Estándar. A `power=1` los puntos lejanos influyen más; a `power=3` influyen menos.

```python
pesos_norm = pesos / pesos.sum(axis=1, keepdims=True)


Se **normalizan los pesos** para que sumen 1 (para que el resultado final sea un promedio ponderado válido).


> **Mapa 8:** Interpolación continua vs reclasificada

> **Lado izquierdo: Superficie continua de NO₃**

**¿Qué muestra?**  
NO₃ interpolada en mg/L, de 0 a 150. Escala: verde (bajo) → rojo (alto).

**¿Por qué?**  
Muestra la **concentración física real** de forma continua. Útil para ver tendencias.

**¿Cómo interpretarlo?**  
Una zona completamente roja indica contaminación severa por nitratos.

> **Lado derecho: Clases reclasificadas**

**¿Qué muestra?**  
La misma superficie continua, pero clasificada en 3 colores: verde (normal), naranja (riesgo), rojo (afectada), según los umbrales (25 y 37.5 mg/L).

**¿Por qué?**  
Facilita la lectura: no necesitas memorizar que 30 mg/L es "riesgo". El color lo dice.

**¿Cómo interpretarlo?**  
La frontera entre verde y naranja = 25 mg/L.  
La frontera entre naranja y rojo = 37.5 mg/L.

**Nota importante**: Esta es una **interpolación**, no una medición. La superficie es suave porque IDW promedia. La realidad es más irregular.

In [16]:
z_clase = np.where(z_no3_masked < UMBRAL_RIESGO, 0,
            np.where(z_no3_masked < UMBRAL_AFECTADA, 1, 2))

fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE_GRANDE)

ax = axes[0]
im = ax.pcolormesh(xx_utm, yy_utm, z_no3_masked, cmap=CMAP_CONTINUO, shading='auto')
gdf_municipios_utm.boundary.plot(ax=ax, color='white', linewidth=0.3)
gdf_ipa_utm.plot(ax=ax, color='black', markersize=5, alpha=0.5, zorder=10)
ax.set_title('Concentración Interpolada de NO₃ (mg/L)', fontsize=12, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, label='NO₃ (mg/L)')

ax = axes[1]
ax.pcolormesh(xx_utm, yy_utm, z_clase, cmap=CMAP_CLASES, norm=NORM_CLASES, shading='auto')
gdf_municipios_utm.boundary.plot(ax=ax, color='white', linewidth=0.3)
gdf_ipa_utm.plot(ax=ax, color='black', markersize=5, alpha=0.5, zorder=10)
ax.set_title('Clases Reclasificadas', fontsize=12, fontweight='bold')

leyenda = [Patch(facecolor=PALETA_COLORES['normal'], label='Normal'),
           Patch(facecolor=PALETA_COLORES['riesgo'], label='Riesgo'),
           Patch(facecolor=PALETA_COLORES['afectada'], label='Afectada')]
fig.legend(handles=leyenda, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Interpolación Espacial: NO₃ Continuo vs. Reclasificado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_08_interpolacion_no3_continuo_vs_clases.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_08_interpolacion_no3_continuo_vs_clases.png")


  ✓ mapa_08_interpolacion_no3_continuo_vs_clases.png


### 6. Serie temporal 2015 - 2025 

**¿Qué se hace?**

Se generan **11 mapas** (uno por año), cada uno con una interpolación IDW usando solo los datos disponibles en ese año.

**¿Por qué?**

Permite ver si hay **cambios a lo largo del tiempo**:
- ¿Las zonas rojas desaparecen (mejora)?
- ¿Nuevas zonas rojas aparecen (empeora)?

**¿Cómo interpretarlo?**

```
2015: 106 observaciones, 50+ puntos IPA utilizados
2018: 50 observaciones (mínimo), cobertura irregular
2025: 205 observaciones (máximo), cobertura más uniforme
```

**Importante**: La malla de 2018 es menos confiable porque hay menos datos. No compares 2018 directamente con 2025 como si fuera una verdadera evolución.

**Lectura visual**: Busca patrones consistentes entre años. Si la zona norte siempre sale roja, es probable contaminación real, no artefacto de interpolación.


In [17]:
anios_disponibles = sorted(coords_completo['anio'].unique())

superficies_anio = {}
for anio in anios_disponibles:
    datos_anio = coords_completo[coords_completo['anio'] == anio]
    if len(datos_anio) >= 4:
        puntos_anio = datos_anio.groupby(['ipa'])[['lat', 'lon', 'no3_mgl']].agg(
            {'lat': 'first', 'lon': 'first', 'no3_mgl': 'mean'}).reset_index()

        gdf_anio = gpd.GeoDataFrame(
            puntos_anio,
            geometry=gpd.points_from_xy(puntos_anio['lon'], puntos_anio['lat']),
            crs=CRS_WGS84
        ).to_crs(CRS_UTM30N)

        pts_este_anio = gdf_anio.geometry.x.values
        pts_norte_anio = gdf_anio.geometry.y.values
        pts_no3_anio = gdf_anio['no3_mgl'].values

        z_anio = idw_interpolar(pts_este_anio, pts_norte_anio, pts_no3_anio,
                               grid_este, grid_norte, power=IDW_POWER,
                               k=min(IDW_K, len(pts_este_anio)))
        z_anio_masked = np.where(mask_utm, z_anio, np.nan).reshape(xx_utm.shape)
        superficies_anio[anio] = z_anio_masked

n_mapas = len(superficies_anio)
n_filas, n_cols = 3, 4
fig, axes = plt.subplots(n_filas, n_cols, figsize=(16, 13))
axes = axes.ravel()

for idx, (anio, z_anio) in enumerate(sorted(superficies_anio.items())):
    ax = axes[idx]
    im = ax.pcolormesh(xx_utm, yy_utm, z_anio, cmap=CMAP_CONTINUO, shading='auto', vmin=0, vmax=150)
    gdf_municipios_utm.boundary.plot(ax=ax, color='white', linewidth=0.2)

    datos_anio = coords_completo[coords_completo['anio'] == anio]
    puntos_anio = datos_anio.groupby(['ipa'])[['lat', 'lon']].first().reset_index()
    gdf_pts_anio = gpd.GeoDataFrame(
        puntos_anio,
        geometry=gpd.points_from_xy(puntos_anio['lon'], puntos_anio['lat']),
        crs=CRS_WGS84
    ).to_crs(CRS_UTM30N)
    gdf_pts_anio.plot(ax=ax, color='black', markersize=3, alpha=0.4, zorder=10)

    ax.set_title(f"{anio} (n={len(puntos_anio)} puntos)", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

for idx in range(n_mapas, len(axes)):
    axes[idx].axis('off')

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax, label='NO₃ (mg/L)')

plt.suptitle('Serie Temporal: Concentración Interpolada de NO₃ (2015–2025)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 0.9, 0.98])
plt.savefig(carpeta_mapas / 'mapa_09_serie_anual_2015_2025.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_09_serie_anual_2015_2025.png")

C:\Users\mcangulo\AppData\Local\Temp\ipykernel_27456\1250129321.py:57: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.98])


  ✓ mapa_09_serie_anual_2015_2025.png


### 7. Incertidumbre espacial

**¿Qué se hace?**

Para cada celda, se calcula la **distancia en km al punto IPA más cercano**. Se colorea: amarillo (cerca, fiable) → rojo (lejos, menos fiable).

**¿Por qué?**

IDW es más confiable cerca de puntos observados. Lejos, es principalmente extrapolación. Este mapa muestra dónde confiar en los resultados.

**¿Cómo interpretarlo?**

**Amarillo** (< 1 km): Muy cercano a un punto medido. Predicción fiable.  
**Rojo** (> 15 km): Muy lejos del punto medido más cercano. Predicción especulativa.

**Decisión práctica**: Si un municipio está completamente rojo, no uses los mapas para tomar decisiones allí sin antes muestrear.


In [18]:
tree = cKDTree(np.column_stack([pts_este, pts_norte]))
dist_minima, _ = tree.query(np.column_stack([grid_este, grid_norte]), k=1)
dist_minimas_masked = np.where(mask_utm, dist_minima, np.nan).reshape(xx_utm.shape)

fig, ax = plt.subplots(figsize=FIG_SIZE_MEDIA)
im = ax.pcolormesh(xx_utm, yy_utm, dist_minimas_masked / 1000, cmap='YlOrRd', shading='auto')
gdf_municipios_utm.boundary.plot(ax=ax, color='white', linewidth=0.3)
gdf_ipa_utm.plot(ax=ax, color='blue', markersize=10, alpha=0.6, zorder=10, label='Puntos IPA')
ax.set_title('Incertidumbre: Distancia al Punto IPA Más Cercano', fontsize=13, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, label='Distancia (km)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(carpeta_mapas / 'mapa_10_incertidumbre_distancia_minima.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print("  ✓ mapa_10_incertidumbre_distancia_minima.png")

  ✓ mapa_10_incertidumbre_distancia_minima.png


### 8. Mapa interactivo folium

**¿Qué se hace?**

Se genera un archivo HTML con **capas activables** por año. Cada punto es un círculo que se puede clickear para ver detalles (IPA, NO₃, clase, predicción).

**¿Por qué?**

Permite exploración interactiva sin abrir Python nuevamente. Útil para presentaciones o discusiones con colegas.

**¿Cómo interpretarlo?**

Abre en navegador:
1. Panel derecho: Marca/desmarca años para verlos.
2. Clickea puntos para ver detalles: "IPA 241050045 | Año 2020 | NO3: 45.2 mg/L | Clase: riesgo | Predicción: afectada".
3. Zoom y pan con ratón.

In [19]:
import folium
from folium import IFrame

# Centro del mapa
centro_lat = coords_completo["lat"].mean()
centro_lon = coords_completo["lon"].mean()

m = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=9,
    tiles="CartoDB positron",
    control_scale=True
)

# Límites municipales
folium.GeoJson(
    gdf_municipios,
    style_function=lambda x: {
        "color": "#808080",
        "weight": 0.6,
        "opacity": 0.7,
        "fillOpacity": 0
    },
    name="Límites municipales"
).add_to(m)


# ------------------------------------------------------------
# Leyenda
# ------------------------------------------------------------

leyenda_html = """
<div style="
    position:fixed;
    bottom:25px;
    left:25px;
    width:250px;
    z-index:9999;
    background-color:rgba(255,255,255,0.96);
    border:1px solid #D1D5DB;
    border-radius:7px;
    box-shadow:0 2px 10px rgba(0,0,0,0.18);
    padding:14px 16px;
    font-family:Arial, sans-serif;
    color:#333;
">

    <div style="
        font-size:15px;
        font-weight:bold;
        margin-bottom:3px;
    ">
        Nivel de contaminación
    </div>

    <div style="
        font-size:11px;
        color:#777;
        margin-bottom:12px;
    ">
        Concentración observada de NO<sub>3</sub><sup>−</sup>
    </div>

    <div style="display:flex; align-items:center; margin-bottom:9px;">
        <span style="
            width:13px;
            height:13px;
            border-radius:50%;
            background:#1D9E75;
            border:1px solid #116149;
            margin-right:9px;
        "></span>

        <span style="font-size:12px;">
            <b>Normal:</b> NO<sub>3</sub><sup>−</sup> &lt; 25 mg/L
        </span>
    </div>

    <div style="display:flex; align-items:center; margin-bottom:9px;">
        <span style="
            width:13px;
            height:13px;
            border-radius:50%;
            background:#EF9F27;
            border:1px solid #B56F10;
            margin-right:9px;
        "></span>

        <span style="font-size:12px;">
            <b>Riesgo:</b> 25–37,5 mg/L
        </span>
    </div>

    <div style="display:flex; align-items:center;">
        <span style="
            width:13px;
            height:13px;
            border-radius:50%;
            background:#E24B4A;
            border:1px solid #A33130;
            margin-right:9px;
        "></span>

        <span style="font-size:12px;">
            <b>Afectada:</b> NO<sub>3</sub><sup>−</sup> ≥ 37,5 mg/L
        </span>
    </div>

    <div style="
        margin-top:12px;
        padding-top:9px;
        border-top:1px solid #E5E7EB;
        font-size:10px;
        color:#888;
    ">
        Seleccione un punto para consultar sus resultados.
    </div>
</div>
"""

m.get_root().html.add_child(folium.Element(leyenda_html))


# ------------------------------------------------------------
# Capas por año
# ------------------------------------------------------------

for anio in anios_disponibles:

    fg = folium.FeatureGroup(
        name=f"Año {anio}",
        show=(anio == anios_disponibles[-1])
    )

    datos_anio = coords_completo[
        coords_completo["anio"] == anio
    ]

    for _, fila in datos_anio.iterrows():

        color = PALETA_COLORES[fila["clase"]]
        color_pred = PALETA_COLORES[fila["clase_predicha"]]

        # Resultado del modelo
        if fila["acierto"] == 1:
            resultado = "Predicción correcta"
            color_resultado = "#1D9E75"
            simbolo = "✓"
        else:
            resultado = "Predicción incorrecta"
            color_resultado = "#E24B4A"
            simbolo = "✕"

        popup_html = f"""
        <html>
        <body style="margin:0; font-family:Arial, sans-serif;">

            <div style="width:275px; color:#333;">

                <div style="
                    background:#F5F6F8;
                    border-bottom:1px solid #DDDDDD;
                    padding:12px;
                ">
                    <div style="font-size:15px; font-weight:bold;">
                        Punto IPA {fila["ipa"]}
                    </div>

                    <div style="
                        margin-top:3px;
                        font-size:11px;
                        color:#777;
                    ">
                        Observación del año {int(fila["anio"])}
                    </div>
                </div>

                <div style="padding:12px;">

                    <div style="
                        margin-bottom:12px;
                        padding-bottom:10px;
                        border-bottom:1px solid #E5E7EB;
                    ">
                        <div style="font-size:11px; color:#777;">
                            Concentración de nitratos
                        </div>

                        <div style="
                            margin-top:3px;
                            font-size:20px;
                            font-weight:bold;
                        ">
                            {fila["no3_mgl"]:.2f}
                            <span style="font-size:12px; font-weight:normal;">
                                mg/L
                            </span>
                        </div>
                    </div>

                    <div style="
                        display:flex;
                        justify-content:space-between;
                        margin-bottom:10px;
                        font-size:12px;
                    ">
                        <span style="color:#777;">Clase observada</span>

                        <span style="font-weight:bold;">
                            <span style="
                                display:inline-block;
                                width:10px;
                                height:10px;
                                border-radius:50%;
                                background:{color};
                                margin-right:5px;
                            "></span>
                            {str(fila["clase"]).capitalize()}
                        </span>
                    </div>

                    <div style="
                        display:flex;
                        justify-content:space-between;
                        margin-bottom:12px;
                        font-size:12px;
                    ">
                        <span style="color:#777;">Clase predicha</span>

                        <span style="font-weight:bold;">
                            <span style="
                                display:inline-block;
                                width:10px;
                                height:10px;
                                border-radius:50%;
                                background:{color_pred};
                                margin-right:5px;
                            "></span>
                            {str(fila["clase_predicha"]).capitalize()}
                        </span>
                    </div>

                    <div style="
                        padding:8px;
                        background:{color_resultado}18;
                        border-left:3px solid {color_resultado};
                        border-radius:3px;
                        color:{color_resultado};
                        font-size:12px;
                        font-weight:bold;
                    ">
                        {simbolo} {resultado}
                    </div>

                    <div style="
                        margin-top:13px;
                        font-size:11px;
                        font-weight:bold;
                    ">
                        Probabilidades del modelo
                    </div>

                    <div style="
                        margin-top:7px;
                        font-size:11px;
                        line-height:1.7;
                        color:#555;
                    ">
                        <div>Normal: {fila["prob_normal"]:.1%}</div>
                        <div>Riesgo: {fila["prob_riesgo"]:.1%}</div>
                        <div>Afectada: {fila["prob_afectada"]:.1%}</div>
                    </div>

                </div>
            </div>

        </body>
        </html>
        """

        # IFrame evita que las etiquetas HTML aparezcan como texto
        iframe = IFrame(
            html=popup_html,
            width=300,
            height=350
        )

        popup = folium.Popup(
            iframe,
            max_width=320
        )

        tooltip = (
            f"IPA {fila['ipa']} | "
            f"{str(fila['clase']).capitalize()} | "
            f"{fila['no3_mgl']:.1f} mg/L"
        )

        folium.CircleMarker(
            location=[fila["lat"], fila["lon"]],
            radius=6,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            opacity=1,
            weight=1.5,
            popup=popup,
            tooltip=tooltip
        ).add_to(fg)

    fg.add_to(m)


# Control de capas
folium.LayerControl(
    collapsed=True,
    position="topright"
).add_to(m)


# Guardar mapa
ruta_html = carpeta_mapas / "mapa_11_interactivo_folium.html"

m.save(str(ruta_html))

print("✓ mapa_11_interactivo_folium.html")

✓ mapa_11_interactivo_folium.html


### 11. Tabla de Predicciones OOF

#### ¿Qué se hace?

Se guarda un CSV con todas las 1172 predicciones, incluyendo:
- IPA, año, clase real, clase predicha, acierto
- NO₃ medido
- Probabilidades del modelo para cada clase

#### ¿Por qué?

Permite análisis posterior sin necesidad de reejecutar el script:
- Filtrar por IPA para ver cómo evolucionó un pozo.
- Estudiar casos donde el modelo se equivocó mucho.
- Exportar a bases de datos o reportes.

#### ¿Cómo interpretarlo?

Abre en Excel o Python:
```
ipa       anio  clase     clase_predicha  acierto  no3_mgl  prob_normal  prob_riesgo  prob_afectada
241050045 2020  normal    normal          1        15.2     0.92         0.07         0.01
211020172 2024  afectada  riesgo          0        42.1     0.05         0.70         0.25
```

- **Fila 1**: Clase verdadera = normal, predicción = normal, acertó. NO₃ = 15.2 (dentro de normal < 25).
- **Fila 2**: Clase verdadera = afectada, predicción = riesgo, erró. Modelo solo 25% seguro de afectada (prob=0.25).

**Uso**: El IPA 211020172 debería revisarse. ¿Por qué el modelo no detectó afectada? Quizá falta información en variables climáticas o agrícolas para ese año.


### 9. Resumen final 

In [20]:
print("\n" + "="*80)
print("GENERACIÓN COMPLETADA CON EXITO")
print("="*80)
print(f"\nMapas guardados en: {carpeta_mapas}")
archivos = sorted(carpeta_mapas.glob('*'))
for i, arch in enumerate(archivos, 1):
    print(f"  {i:2d}. {arch.name}")

print(f"\nTotal: {len(archivos)} archivos generados")
print("\n✓ Notebook ejecutado exitosamente")


GENERACIÓN COMPLETADA CON EXITO

Mapas guardados en: C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\12_resultados_mapas
   1. mapa_01_red_monitoreo.png
   2. mapa_02_densidad_observaciones.png
   3. mapa_03_concentracion_no3_observada.png
   4. mapa_04_clase_observada_ipa.png
   5. mapa_05_real_vs_predicho_oof.png
   6. mapa_06_errores_modelo.png
   7. mapa_07_zonas_vulnerables_comparacion.png
   8. mapa_08_interpolacion_no3_continuo_vs_clases.png
   9. mapa_09_serie_anual_2015_2025.png
  10. mapa_10_incertidumbre_distancia_minima.png
  11. mapa_11_interactivo_folium.html
  12. predicciones_oof_completas.csv

Total: 12 archivos generados

✓ Notebook ejecutado exitosamente


### Limitaciones Importantes

#### Sobre Interpolación

- **No es medición directa**: IDW estima basada en puntos observados. Si todos los puntos en una zona tienen NO₃ bajo, IDW estimará bajo, aunque la realidad podría ser distinta en áreas no muestreadas.
- **Depende de densidad de puntos**: Zonas con pocos puntos tienen interpolación menos confiable (ver Mapa 10).

#### Sobre Predicciones

- **Caída en generalización**: El modelo tiene 80% accuracy en los datos donde se entrenó (1172 obs), pero solo ~60% en pozos nuevos (validación espacial). No confíes ciegamente en predicciones fuera del área observada.
- **Data leakage evitado**: Usamos OOF correctamente, así que las métricas son realistas.

#### Sobre Series Temporales

- **Cobertura variable**: 2018 tiene solo 50 observaciones vs. 205 en 2025. Los mapas de 2018 son menos confiables.
- **No es causalidad**: Un cambio en los mapas entre 2018 y 2025 podría deberse a variación natural, cambios en políticas agrícolas, o simplemente más puntos muestreados en 2025.

---

### ¿Qué Hacer Con Estos Mapas?

### En la Memoria TFM

Incorpora:
- Mapas 1–6 como figuras en la sección metodológica (explicar red, observaciones, modelo).
- Mapa 8 como figura principal en resultados (interpolación).
- Mapa 9 como anexo (serie temporal).
- CSV como anexo (tabla de predicciones).

### Decisiones Futuras

- **No uses los mapas como delimitación oficial** de zonas vulnerables. La CHE es la autoridad.
- **Considera más muestreo** en áreas completamente rojas en el Mapa 10 (alta incertidumbre).
- **Busca explicaciones** para errores del modelo en zonas específicas (Mapa 6).

---

## Preguntas Frecuentes

**P: ¿Por qué 101 puntos IPA y no más?**  
R: Esos son los puntos disponibles en La Rioja. No es decisión del análisis, es limitación de datos.

**P: ¿El modelo predice bien?**  
R: 80% accuracy es bueno, pero no perfecto. F1 para la clase "riesgo" es solo 0.65, así que cuidado con falsos negativos.

**P: ¿Puedo interpolar fuera de La Rioja?**  
R: No. Los mapas están recortados al contorno de La Rioja. IDW extrapolará mal fuera del área de puntos observados.

**P: ¿Cambió el riesgo entre 2015 y 2025?**  
R: Los mapas sugieren tendencias, pero **no lo afirmes sin análisis estadístico adicional**. La interpolación es suave; no captura variabilidad fina.

**P: ¿Qué hago si un municipio tiene error rojo (Mapa 6)?**  
R: Revisa los datos de entrada para ese municipio. Quizá hay variables incorrectas. Si los datos son correctos, considera que el modelo tiene limitaciones en esa zona.